In [ ]:
# Register notebook import hook for loading .ipynb files as modules
import sys

class NotebookFinderAndLoader:
    def find_spec(self, fullname, path, target=None):
        import os
        from importlib.machinery import ModuleSpec
        name = fullname.rsplit('.', 1)[-1]
        search_paths = path if path else [os.getcwd()]
        for p in search_paths:
            nb_path = os.path.join(p, name + '.ipynb')
            if os.path.isfile(nb_path):
                return ModuleSpec(fullname, self, origin=nb_path)
        return None

    def create_module(self, spec):
        return None

    def exec_module(self, module):
        import json
        with open(module.__spec__.origin, 'r', encoding='utf-8') as f:
            nb = json.load(f)
        for cell in nb['cells']:
            if cell['cell_type'] == 'code':
                exec(''.join(cell['source']), module.__dict__)

sys.meta_path.append(NotebookFinderAndLoader())


In [ ]:
# main.py — Movie Recommendation System
# Roll No: CSE003 & CSE049 | CSE Mini Project | 2024-2025
# Run: python main.py

import os, sys
from preprocessing  import load_and_preprocess
from recommendation import build_model, recommend

# Check dataset files exist
for f in ["tmdb_5000_movies.csv", "tmdb_5000_credits.csv"]:
    if not os.path.isfile(f):
        print(f"ERROR: '{f}' not found.")
        print("Download from: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata")
        sys.exit()

print("Loading data...")
df = load_and_preprocess()

print("Building model...")
similarity = build_model(df)
print(f"Done. {len(df)} movies loaded.\n")

# Interactive loop
while True:
    movie = input("Enter movie name (or 'quit' to exit): ").strip()
    if movie.lower() in ('quit', 'exit', ''):
        break
    
    recs, suggestions = recommend(movie, df, similarity)
    if recs:
        print("\nTop 5 Recommended Movies:")
        for i, title in enumerate(recs, 1):
            print(f"{i}. {title}")
    else:
        print(f"Movie '{movie}' not found in dataset.")
        if suggestions:
            print(f"Did you mean: {suggestions}")
